# Can the geocentric alignment identify an eclipse day?

Consider the directions to the Sun and Moon as seen from the center of Earth. If the directions are far apart throughout the day, the lunar shadow cannot reach Earth. Near alignment, the answer also depends on the position of the observer on its surface. We let a classifier learn how the measured alignment and positions distinguish the catalog labels.

Use `df_jpl_complete.csv` and `dataset_A_daily.csv` from Data Cleaning. The last example makes two Horizons requests per date.

In [ ]:
%pip install numpy pandas requests scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import requests
df_jpl=pd.read_csv("df_jpl_complete.csv")
df_jpl.head()

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

## Measuring alignment

Write $\mathbf r_S$ and $\mathbf r_M$ for the geocentric position vectors in km. The angle at Earth is
$$\alpha=\cos^{-1}\!\left(\frac{\mathbf r_S\cdot\mathbf r_M}{\|\mathbf r_S\|\,\|\mathbf r_M\|}\right).$$
The quotient is dimensionless. We clip it to the domain of the inverse cosine to absorb roundoff and express the result in degrees.

In [ ]:
import re

def date_to_jdn(date):
    y, m, d = map(
        int,
        re.fullmatch(
            r"([+-]?\d+)-(\d{1,2})-(\d{1,2})",
            str(date).split()[0]
        ).groups()
    )

    a = (14 - m) // 12
    y = y + 4800 - a
    m = m + 12*a - 3

    return d + (153*m + 2)//5 + 365*y + y//4 - y//100 + y//400 - 32045


# Data preparation
df_train = df_jpl.copy()

position_cols = [
    "x_km_sun", "y_km_sun", "z_km_sun",
    "x_km_moon", "y_km_moon", "z_km_moon"
]

sun = df_train[[
    "x_km_sun", "y_km_sun", "z_km_sun"
]].to_numpy()

moon = df_train[[
    "x_km_moon", "y_km_moon", "z_km_moon"
]].to_numpy()

dot_product = np.sum(sun * moon, axis=1)

norms = np.linalg.norm(sun, axis=1) * np.linalg.norm(moon, axis=1)

cos_angle = np.clip(dot_product / norms, -1, 1)

df_train["angle_deg"] = np.degrees(np.arccos(cos_angle))

df_train["day_number"] = df_train["sample_day"].apply(date_to_jdn)

One training example represents one sampled day. We average each Cartesian component and keep the **smallest observed angle**. Averaging the angles would blur the few observations nearest conjunction. The Julian day number supplies the eighth input.

In [ ]:
# Group by day and retain the minimum separation
df_train = df_train.groupby("day_number", as_index=False).agg({
    **{col: "mean" for col in position_cols},
    "angle_deg": "min",
    "eclipse": "max"
})

df_train = df_train.dropna()
df_train = df_train.sort_values("day_number").reset_index(drop=True)

# Features and target
features = ["day_number"] + position_cols + ["angle_deg"]

X = df_train[features]
y = df_train["eclipse"].astype(int)

## Testing on later dates

Dates are sorted before splitting. We fit successive shallow corrections to the classifier score using histogram gradient boosting on the first 80% and predict the remaining 20%. In the matrix, rows are catalog labels and columns are predictions. A false negative is particularly important here because it prevents the later trajectory calculation.

In [ ]:
# Chronological split: 80% past, 20% future
split = int(len(df_train) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Model
model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

# False negatives
errors = df_train.iloc[split:].copy()

errors["prediction"] = y_pred

false_negatives = errors[
    (errors["eclipse"] == 1) &
    (errors["prediction"] == 0)
]

print(false_negatives[
    ["day_number", "angle_deg", "eclipse", "prediction"]
])

The recorded holdout contained 131 negative days and 27 eclipse days. All 158 were classified correctly. These days were sampled with the same window-selection rule as the training set. What happens if the only information available to us is a date?

In [ ]:
import random
rng = random.Random(42)
df_A_daily=pd.read_csv("dataset_A_daily.csv")
yesdates=df_A_daily[df_A_daily["eclipse"]==1]["date"].tolist()
nodates=df_A_daily[df_A_daily["eclipse"]==0]["date"].tolist()
yes_dates=rng.sample(yesdates, 25)
no_dates=rng.sample(nodates, 75)

## Predicting from a calendar date alone

For a new date we do not know when closest approach occurs. We request 96 positions per body, spaced by 15 minutes from midnight through 23:45 UT. This changes both the span over which we average positions and the precision of the minimum angle.

In [ ]:
import requests
import csv
import time

url = "https://ssd.jpl.nasa.gov/api/horizons.api"

features = [
    "day_number",
    "x_km_sun", "y_km_sun", "z_km_sun",
    "x_km_moon", "y_km_moon", "z_km_moon",
    "angle_deg"
]


def get_positions(body_id, day):

    start = day - 0.5
    stop = start + 23.75 / 24

    params = {
        "format": "text",
        "COMMAND": f"'{body_id}'",
        "OBJ_DATA": "'NO'",
        "MAKE_EPHEM": "'YES'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": "'500@399'",
        "REF_PLANE": "'FRAME'",
        "OUT_UNITS": "'KM-S'",
        "VEC_TABLE": "'1'",
        "VEC_LABELS": "'NO'",
        "CSV_FORMAT": "'YES'",
        "TIME_TYPE": "'UT'",
        "START_TIME": f"'JD{start:.8f}'",
        "STOP_TIME": f"'JD{stop:.8f}'",
        "STEP_SIZE": "'15m'"
    }

    response = requests.get(url, params=params, timeout=90)
    response.raise_for_status()

    text = response.text

    if "$$SOE" not in text or "$$EOE" not in text:
        raise ValueError(f"JPL error:\n{text[:1000]}")

    data = text.split("$$SOE")[1].split("$$EOE")[0]
    rows = list(csv.reader(data.strip().splitlines()))

    positions = np.array([
        [float(x) for x in row[2:5]]
        for row in rows
    ])

    if len(positions) != 96:
        raise ValueError(f"Expected 96 observations, got {len(positions)}")

    return positions
def predict_eclipses(dates):

    df = pd.DataFrame({"date": dates})
    df["day_number"] = df["date"].apply(date_to_jdn)

    results = []
    hours = np.arange(0, 24, 0.25)

    print("Observations per day:", len(hours))

    for i in range(len(df)):

        day = df.iloc[i]["day_number"]

        sun_day = get_positions("10", day)
        moon_day = get_positions("301", day)

        dot = np.sum(sun_day * moon_day, axis=1)

        norms = (
            np.linalg.norm(sun_day, axis=1) *
            np.linalg.norm(moon_day, axis=1)
        )

        if np.any(norms == 0):
            raise ValueError("Invalid position vector")

        angles = np.degrees(
            np.arccos(np.clip(dot / norms, -1, 1))
        )

        results.append({
            "date": df.iloc[i]["date"],
            "day_number": day,
            "x_km_sun": sun_day[:, 0].mean(),
            "y_km_sun": sun_day[:, 1].mean(),
            "z_km_sun": sun_day[:, 2].mean(),
            "x_km_moon": moon_day[:, 0].mean(),
            "y_km_moon": moon_day[:, 1].mean(),
            "z_km_moon": moon_day[:, 2].mean(),
            "angle_deg": angles.min()
        })

        print(f"Processed {i+1}/{len(df)} dates")

        time.sleep(1)

    result = pd.DataFrame(results)

    X_new = result[features]

    result["prediction"] = model.predict(X_new)
    result["probability"] = model.predict_proba(X_new)[:, 1]

    return result

We draw 25 positive and 75 negative dates from the daily catalog. The prediction loop preserves their order, so the reference labels must follow that same order. This is a check with newly queried features on a selected sample. Its class proportions differ from those of the full calendar.

In [ ]:
# External evaluation
dates = yes_dates+no_dates
y_true = [1]*len(yes_dates) + [0]*len(no_dates)

results = predict_eclipses(dates)

results["eclipse"] = y_true

print(confusion_matrix(y_true, results["prediction"]))

print(classification_report(
    y_true,
    results["prediction"],
    zero_division=0
))

display(results[[
    "date",
    "eclipse",
    "prediction",
    "probability",
    "angle_deg"
]])

The original external run recorded progress through 85 of the 100 dates and did not retain a final confusion matrix. The perfect score above belongs only to the chronological holdout. Running the last cell to completion produces the separate catalog-sample evaluation.